# Naive Bayes TF BOW Binary Dimensions
This notebook trains four binary Multinomial Naive Bayes classifiers (one per MBTI dimension).

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path
from sklearn.feature_extraction.text import CountVectorizer  # <--- The key difference!
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from imblearn.pipeline import Pipeline as ImbPipeline

 --- STEP 1: Load Data  ---


In [13]:
print(" Loading data...")
current_dir = Path.cwd()
csv_path = None
for parent in [current_dir] + list(current_dir.parents)[:3]:
    candidate = parent / 'data' / 'processed' / 'preprocessed_data.csv'
    if candidate.exists():
        csv_path = candidate
        break
if not csv_path:
    csv_path = Path(r"C:\Users\piete\Documents\ML_MBTI_project\data\processed\preprocessed_data.csv")

df = pd.read_csv(csv_path)
df = df.dropna(subset=['posts', 'type'])

 Loading data...


 --- STEP 2: Vectorization (Bag of Words) ---

In [14]:
df = pd.read_csv(csv_path)
print(f"   Columns found: {df.columns.tolist()}")
text_col = 'posts' 
target_col = 'type'

# Remove rows where text or type is missing
df = df.dropna(subset=[text_col, target_col]) 

#Do NOT vectorize here. 
# We just pass the raw text list to the next step.
print("\n Preparing raw text for pipeline...")
X = df[text_col].values  # X is now a list of strings, not a matrix
y = df[target_col].values

print(f"Data shape: {X.shape[0]} rows")

   Columns found: ['type', 'posts', 'num_posts', 'no. of. words', 'type_encoded']

 Preparing raw text for pipeline...
Data shape: 8462 rows


--- STEP 3: Train & Evaluate  ---


In [15]:
print("\n Starting Naive Bayes (BoW) with Grid Search Tuning")

# Assuming X (raw text list) and y (full labels) are already loaded from Step 1 & 2

dimensions = [('I', 'E'), ('N', 'S'), ('T', 'F'), ('J', 'P')]
results = {}
best_params_log = {}

# Set up 5-fold cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Pipeline for Bag of Words
pipeline = ImbPipeline([
    ('vect', CountVectorizer(stop_words='english')), 
    ('sampler', RandomOverSampler(random_state=42)),
    ('classifier', MultinomialNB())
])

param_grid = {
    'vect__max_features': [5000],      # removed None/2000 to keep consistent and save computation
    'classifier__alpha': [0.01, 1.0, 5.0, 10.0],   # BoW often likes higher alpha
    'classifier__fit_prior': [True, False]
}

sns.set_style("whitegrid")

for trait1, trait2 in dimensions:
    print(f"\n--- Tuning {trait1} vs {trait2} (BoW) ---")
    
    # Create binary labels
    y_binary = np.array([1 if trait2 in label else 0 for label in y])
    
    # Initialize GridSearchCV
    # ADDED: return_train_score=True
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=cv,
        scoring='f1_macro', 
        n_jobs=-1,          
        verbose=1,
        return_train_score=True 
    )
    
    grid_search.fit(X, y_binary)
    
    # Store best results
    best_score = grid_search.best_score_
    best_params = grid_search.best_params_
    results[f"{trait1}/{trait2}"] = best_score
    best_params_log[f"{trait1}/{trait2}"] = best_params
    
    #Calculate and Print Overfitting Gap
    best_index = grid_search.best_index_
    train_score = grid_search.cv_results_['mean_train_score'][best_index]
    test_score = grid_search.cv_results_['mean_test_score'][best_index]
    gap = train_score - test_score

#     print(f"Best F1-Score: {best_score:.4f}")
#     print(f"Best Params: {best_params}")
#     print(f" Training F1: {train_score:.4f}")
#     print(f" Test F1:     {test_score:.4f}")
#     print(f" Overfitting Gap: {gap:.2%} ({'HIGH' if gap > 0.10 else 'OK'})")
    
#     # Generate predictions on full dataset using the best model found
#     y_pred_full = grid_search.predict(X)
#     print(f"   Confusion Matrix (Full Data):\n{confusion_matrix(y_binary, y_pred_full)}")

# #PLOTTING THE RESULTS
# print("\n Generating Bar Graph of Best Scores (BoW)...")
# plt.figure(figsize=(10, 6))
# keys = list(results.keys())
# values = list(results.values())

# # Create bars
# bars = plt.bar(keys, values, color=['#4c72b0', '#55a868', '#c44e52', "#ffe62a"])

# plt.title('Optimized Naive Bayes (BoW) F1-Score', fontsize=16, pad=20)
# plt.xlabel('Personality Dimension Pair', fontsize=12)
# plt.ylabel('Best Mean F1-Score', fontsize=12)
# plt.ylim(0, 1.0)

# # Add text labels on top of bars
# for bar in bars:
#     height = bar.get_height()
#     plt.text(bar.get_x() + bar.get_width()/2.0, height,
#              f'{height:.2f}', 
#              ha='center', va='bottom', fontsize=12, fontweight='bold')

# plt.tight_layout()
# plt.show()

# #FINAL SUMMARY
# print("\n Optimization Summary (BoW):")
# for dim, params in best_params_log.items():
#     print(f"  {dim}: {params}")

# avg_f1 = sum(results.values()) / len(results)
# print(f"\n Average Optimized F1-Score: {avg_f1:.4f}")

In [16]:
print("\n Starting Naive Bayes (BoW) - Optimizing for STABILITY (Low Overfitting)...")

dimensions = [('I', 'E'), ('N', 'S'), ('T', 'F'), ('J', 'P')]
results_stable = {}
best_params_stable = {}
results_detailed = {}  # For storing final models and metrics

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Pipeline for Bag of Words
pipeline = ImbPipeline([
    ('vect', CountVectorizer(stop_words='english')), 
    ('sampler', RandomOverSampler(random_state=42)),
    ('classifier', MultinomialNB())
])

# We test high Alpha values here to force stability
param_grid = {
    'vect__max_features': [5000],
    'classifier__alpha': [0.1, 1.0, 5.0, 10.0, 20.0],
    'classifier__fit_prior': [True, False]
}

for trait1, trait2 in dimensions:
    dim_name = f"{trait1}/{trait2}"
    print(f"\n--- Tuning {dim_name} (BoW) for Stability ---")
    
    y_binary = np.array([1 if trait2 in label else 0 for label in y])
    
    # Run GridSearchCV with train scores
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=cv,
        scoring='f1_macro',
        n_jobs=-1,
        verbose=1,
        return_train_score=True
    )
    
    # Fit on RAW text (pipeline handles vectorization + upsampling)
    grid_search.fit(X, y_binary)
    
    # Extract GridSearchCV results
    cv_results = grid_search.cv_results_
    mean_test_scores = cv_results['mean_test_score']
    mean_train_scores = cv_results['mean_train_score']
    params = cv_results['params']
    
    # Find the "Most Stable" Parameter Set
    best_stable_gap = float('inf')
    best_stable_params = None
    best_stable_score = 0
    
    # Filter for models that are at least 90% as good as the absolute best F1
    max_f1 = np.max(mean_test_scores)
    threshold = max_f1 * 0.90 
    
    for i in range(len(params)):
        test_score = mean_test_scores[i]
        
        if test_score > threshold:  # Only check "good" models
            train_score = mean_train_scores[i]
            gap = train_score - test_score 
            
            if gap < best_stable_gap:
                best_stable_gap = gap
                best_stable_params = params[i]
                best_stable_score = test_score

    # Save stability results
    results_stable[dim_name] = best_stable_score
    best_params_stable[dim_name] = best_stable_params
    
    print(f"   Selected Alpha for Stability: {best_stable_params['classifier__alpha']}")
    print(f"   CV F1-Score: {best_stable_score:.4f}")
    print(f"   Overfitting Gap: {best_stable_gap:.4f} ({(best_stable_gap*100):.2f}%)")
    
    # ========================================================================
    # Extract metrics directly from GridSearchCV (NO re-training)
    # ========================================================================
    print(f"\n   Extracting comprehensive metrics from GridSearchCV...")
    
    # Simple 80/20 split for held-out test evaluation
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_binary, test_size=0.20, random_state=42, stratify=y_binary
    )
    
    # Get the best model from GridSearchCV (already trained with best params)
    best_model = grid_search.best_estimator_
    
    # Predictions using GridSearchCV's best model
    train_preds = best_model.predict(X_train)
    test_preds = best_model.predict(X_test)
    
    # Calculate comprehensive metrics
    train_acc = accuracy_score(y_train, train_preds)
    test_acc = accuracy_score(y_test, test_preds)
    train_f1 = f1_score(y_train, train_preds, average='macro')
    test_f1 = f1_score(y_test, test_preds, average='macro')
    precision = precision_score(y_test, test_preds, average='macro', zero_division=0)
    recall = recall_score(y_test, test_preds, average='macro', zero_division=0)
    overfit_gap = train_acc - test_acc
    
    # CV score comes from GridSearchCV
    cv_f1_score = best_stable_score
    
    # Extract components from the best pipeline
    vectorizer = best_model.named_steps['vect']
    clf = best_model.named_steps['classifier']
    
    # Store everything in compatible format
    results_detailed[dim_name] = {
        'model': clf,
        'vectorizer': vectorizer,
        'pipeline': best_model,  # Full pipeline for predictions
        'accuracy': 100 * test_acc,
        'train_accuracy': 100 * train_acc,
        'f1_score': 100 * test_f1,
        'train_f1': 100 * train_f1,
        'precision': 100 * precision,
        'recall': 100 * recall,
        'overfit_gap': 100 * overfit_gap,
        'cv_score': 100 * cv_f1_score,
        'best_params': best_stable_params,
        'optimization_type': 'stability',
        'y_test': y_test,
        'y_pred': test_preds
    }
    
    print(f"   ✓ Test Acc: {test_acc:.2%}, F1: {test_f1:.2%}, Overfit Gap: {overfit_gap:.2%}")

# Summary 
print("\n" + "=" * 80)
print("OPTIMIZATION COMPLETE")
print("=" * 80)
print("Stability-optimized parameters (low overfitting):")
for dim, params in best_params_stable.items():
    print(f"  {dim}: alpha={params['classifier__alpha']}, fit_prior={params['classifier__fit_prior']}")
print("=" * 80)


 Starting Naive Bayes (BoW) - Optimizing for STABILITY (Low Overfitting)...

--- Tuning I/E (BoW) for Stability ---
Fitting 5 folds for each of 10 candidates, totalling 50 fits
   Selected Alpha for Stability: 20.0
   CV F1-Score: 0.6144
   Overfitting Gap: 0.0875 (8.75%)

   Extracting comprehensive metrics from GridSearchCV...
   ✓ Test Acc: 75.61%, F1: 70.36%, Overfit Gap: -0.44%

--- Tuning N/S (BoW) for Stability ---
Fitting 5 folds for each of 10 candidates, totalling 50 fits
   Selected Alpha for Stability: 20.0
   CV F1-Score: 0.5935
   Overfitting Gap: 0.1063 (10.63%)

   Extracting comprehensive metrics from GridSearchCV...
   ✓ Test Acc: 80.51%, F1: 70.29%, Overfit Gap: 0.45%

--- Tuning T/F (BoW) for Stability ---
Fitting 5 folds for each of 10 candidates, totalling 50 fits
   Selected Alpha for Stability: 20.0
   CV F1-Score: 0.7386
   Overfitting Gap: 0.0401 (4.01%)

   Extracting comprehensive metrics from GridSearchCV...
   ✓ Test Acc: 80.39%, F1: 80.27%, Overfit Gap: 

--- STEP 4: Save Models ---

Save the stability-optimized models with comprehensive metrics for model comparison.

In [17]:
print("\n" + "=" * 80)
print("SAVING STABILITY-OPTIMIZED MODELS")
print("=" * 80)

# Prepare model data in compatible format
model_data = {
    'dimensions': list(results_detailed.keys()),
    'models': {dim: results_detailed[dim]['model'] for dim in results_detailed},
    'vectorizers': {dim: results_detailed[dim]['vectorizer'] for dim in results_detailed},
    'pipelines': {dim: results_detailed[dim]['pipeline'] for dim in results_detailed},
    'accuracies': {dim: results_detailed[dim]['accuracy'] / 100 for dim in results_detailed},
    
    # Comprehensive metrics from GridSearchCV
    'metrics': {dim: {
        'accuracy': results_detailed[dim]['accuracy'],
        'f1_score': results_detailed[dim]['f1_score'],
        'train_f1': results_detailed[dim]['train_f1'],
        'precision': results_detailed[dim]['precision'],
        'recall': results_detailed[dim]['recall'],
        'train_accuracy': results_detailed[dim]['train_accuracy'],
        'overfit_gap': results_detailed[dim]['overfit_gap'],
        'cv_score': results_detailed[dim]['cv_score']
    } for dim in results_detailed},
    
    # Stability optimization results
    'best_params': {dim: best_params_stable[dim] for dim in best_params_stable},
    'gridsearch_stable_f1': {dim: results_stable[dim] for dim in results_stable},
    'gridsearch_stable_params': {dim: best_params_stable[dim] for dim in best_params_stable},
    
    'optimization_type': 'stability',
    'model_type': 'Naive Bayes (Bag of Words - Stability Optimized)',
    'hyperparameters': {
        'vectorizer': 'CountVectorizer',
        'max_features': 5000,
        'stop_words': 'english',
        'upsampling': True,
        'optimization': 'Stability (Low Overfitting)',
        'selection_criteria': 'Minimum overfitting gap among models within 90% of best F1'
    }
}

# Save
models_dir = Path('../models')
models_dir.mkdir(exist_ok=True)
save_path = models_dir / 'naive_bayes_BOW.pkl'
joblib.dump(model_data, save_path)

print(f"\n✅ Models saved to: {save_path}")

# Print summary
print(f"\n{'=' * 80}")
print("SAVED MODELS SUMMARY")
print("=" * 80)

for dim in results_detailed.keys():
    metrics = model_data['metrics'][dim]
    params = model_data['best_params'][dim]
    
    print(f"\n{dim}:")
    print(f"  Alpha: {params['classifier__alpha']}, Max Features: {params['vect__max_features']}")
    print(f"  Test Acc: {metrics['accuracy']:.2f}%, F1: {metrics['f1_score']:.2f}%, Overfit: {metrics['overfit_gap']:.2f}%")

# Overall averages
avg_acc = np.mean([model_data['metrics'][dim]['accuracy'] for dim in results_detailed])
avg_f1 = np.mean([model_data['metrics'][dim]['f1_score'] for dim in results_detailed])

print(f"\n{'=' * 80}")
print(f"Average Test Accuracy:  {avg_acc:.2f}%")
print(f"Average Test F1-Score:  {avg_f1:.2f}%")
print("=" * 80)
print("✅ All stability-optimized models saved successfully")


SAVING STABILITY-OPTIMIZED MODELS

✅ Models saved to: ..\models\naive_bayes_BOW.pkl

SAVED MODELS SUMMARY

I/E:
  Alpha: 20.0, Max Features: 5000
  Test Acc: 75.61%, F1: 70.36%, Overfit: -0.44%

N/S:
  Alpha: 20.0, Max Features: 5000
  Test Acc: 80.51%, F1: 70.29%, Overfit: 0.45%

T/F:
  Alpha: 20.0, Max Features: 5000
  Test Acc: 80.39%, F1: 80.27%, Overfit: -2.87%

J/P:
  Alpha: 20.0, Max Features: 5000
  Test Acc: 69.76%, F1: 69.12%, Overfit: -1.11%

Average Test Accuracy:  76.57%
Average Test F1-Score:  72.51%
✅ All stability-optimized models saved successfully
